In [0]:
%md
# Customer Ingestion Pipeline (Prod-Grade Pattern)

Ingests customer records from the built-in `samples.bakehouse.sales_customers` sample table into a governed Delta table using an **Extract → Validate → Load (merge) → Verify** pattern.

Unlike a scratch notebook, this version is parameterized per environment, enforces an explicit target schema, upserts idempotently, and fails fast on data quality problems.

In [0]:
# Parameters -- resolved from widgets so this notebook runs unchanged across dev/stage/prod.
# Do not hardcode catalog/schema names in the cells below; read them from these variables instead.
dbutils.widgets.text("target_catalog", "workspace", "Target catalog")
dbutils.widgets.text("target_schema", "bronze_bakehouse", "Target schema")
dbutils.widgets.text("source_table", "samples.bakehouse.sales_customers", "Source table (fully qualified)")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
source_table = dbutils.widgets.get("source_table")
target_table = f"{target_catalog}.{target_schema}.customers"

print(f"Source table: {source_table}")
print(f"Target table: {target_table}")

In [0]:
import logging

logger = logging.getLogger("customer_ingestion")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(handler)

In [0]:
%md
## 1. Extract & validate source

Fail fast if the source table is missing or empty, rather than silently writing zero/partial rows downstream.

In [0]:
source_df = spark.read.table(source_table)

if "customerID" not in source_df.columns:
    raise ValueError(f"Source table {source_table} is missing the expected 'customerID' key column")

source_count = source_df.count()
if source_count == 0:
    raise ValueError(f"Source table {source_table} returned zero rows -- aborting load")

logger.info(f"Read {source_count} rows from {source_table}")

In [0]:
%md
## 2. Define the target schema explicitly

An explicit `CREATE TABLE` with typed columns, a primary key hint, and a table comment -- instead of letting `saveAsTable` infer a schema from whatever shape the source happens to be in on a given run.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ${target_catalog}.${target_schema};

CREATE TABLE IF NOT EXISTS ${target_catalog}.${target_schema}.customers (
  customerID     STRING NOT NULL,
  first_name     STRING,
  last_name      STRING,
  _ingested_at   TIMESTAMP,
  _source_table  STRING,
  CONSTRAINT customers_pk PRIMARY KEY (customerID)
) USING DELTA
COMMENT 'Governed customer dimension, upserted from bakehouse sample data. Owner: data-eng.'

In [0]:
%md
## 3. Load -- idempotent upsert (MERGE)

`MERGE` instead of `overwrite` means re-running this notebook -- retry, backfill, or a scheduled job -- never duplicates rows or destroys table history.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

staged_df = (
    source_df
    .select("customerID", "first_name", "last_name")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_table", F.lit(source_table))
)

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(staged_df.alias("s"), "t.customerID = s.customerID")
    .whenMatchedUpdate(set={
        "first_name": "s.first_name",
        "last_name": "s.last_name",
        "_ingested_at": "s._ingested_at",
        "_source_table": "s._source_table",
    })
    .whenNotMatchedInsertAll()
    .execute()
)

logger.info(f"Merge complete into {target_table}")

In [0]:
%md
## 4. Verify -- post-load data quality gate

Reconcile row counts and check for unexpected nulls before declaring the run successful. A failed check raises, so an orchestrated job run is marked failed instead of silently succeeding on bad data.

In [0]:
target_df = spark.read.table(target_table)
target_count = target_df.count()
null_keys = target_df.filter(F.col("customerID").isNull()).count()

if null_keys > 0:
    raise ValueError(f"{null_keys} rows in {target_table} have a null customerID -- data quality check failed")

if target_count < source_count:
    raise ValueError(
        f"Row count regression: source had {source_count}, target has {target_count} after merge"
    )

logger.info(f"Data quality checks passed: {target_count} rows in {target_table}")

In [0]:
%sql
SELECT * FROM ${target_catalog}.${target_schema}.customers ORDER BY customerID LIMIT 20

In [0]:
%md
## Operationalizing this notebook

- Schedule it as a **Databricks Workflow** task with `target_catalog` / `target_schema` set per environment, instead of hardcoding values or relying on `USE CATALOG`.
- Add a job-level **failure alert/webhook** so a raised data quality check pages someone instead of failing silently.
- Run `OPTIMIZE` and a retention-aware `VACUUM` on `${target_catalog}.${target_schema}.customers` on a separate maintenance schedule, rather than ad hoc from this notebook.
- Replace the full-table `count()` reconciliation with an incremental watermark (e.g. `_ingested_at`) once source volume makes full counts expensive.